In [12]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [13]:
import struct
import math
import time
import json
import numpy as np
from tqdm import tqdm
from scipy.ndimage import binary_dilation 

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [14]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda:0


In [15]:
def get_gpu_memory_usage_mb():
    try:
        if torch.cuda.is_available():
            return torch.cuda.memory_allocated() / (1024 * 1024)
        return 0
    except Exception:
        return 0


In [ ]:

def _concat_points(strokes):
    if len(strokes) == 0:
        return np.zeros((0, 2), dtype=np.float32), []
    idx, all_pts, start = [], [], 0
    for s in strokes:
        s = np.asarray(s, dtype=np.float32)  
        L = len(s)
        if L == 0:
            continue
        all_pts.append(s)
        idx.append((start, start + L))
        start += L
    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32), []
    return np.concatenate(all_pts, axis=0), idx

def _rotate_pts_about_anchor(P, theta, anchor):
    c, s = np.cos(theta), np.sin(theta)
    xy, anc_xy = P[:, :2], anchor[:2]
    D = xy - anc_xy
    x_new = c * D[:, 0] - s * D[:, 1]
    y_new = s * D[:, 0] + c * D[:, 1]
    R = np.stack([x_new, y_new], axis=1) + anc_xy
    if P.shape[1] > 2:
        return np.concatenate([R, P[:, 2:]], axis=1)
    return R

def apply_hanging_on_strokes(strokes, mode="SC"):
    if (mode is None) or (mode == "None") or (len(strokes) == 0):
        return strokes
    P, spans = _concat_points(strokes)
    if P.shape[0] < 2:
        return strokes
    first_pt = P[0]
    if mode == "SC":  
        center = P[:, :2].mean(axis=0)
        v = center - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]
    elif mode == "SE": 
        last_pt = P[-1]
        v = last_pt[:2] - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]
    return strokes

def read_pot_file(filename):
    if not os.path.exists(filename):
        return [], ""
    raw_strokes = []
    with open(filename, "rb") as f:
        while True:
            sample_size = f.read(2)
            if sample_size == b'':
                break
            dword_code = f.read(2)
            if len(dword_code) < 2:
                break
            if dword_code[0] != 0:
                dword_code = bytes((dword_code[1], dword_code[0]))
            tag_code = struct.unpack(">H", dword_code)[0]
            f.read(2)
            try:
                tag = struct.pack('>H', tag_code).decode("gbk")[0]
            except Exception:
                tag = str(tag_code)
            f.read(2)
            strokes_samples, stroke_samples = [], []
            while True:
                bx, by = f.read(2), f.read(2)
                if bx == b'' or by == b'':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\xff\xff':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\x00\x00':
                    strokes_samples.append(stroke_samples)
                    stroke_samples = []
                    continue
                x = struct.unpack("<H", bx)[0]
                y = struct.unpack("<H", by)[0]
                stroke_samples.append((x, y))
            raw_strokes.append(strokes_samples)
    return raw_strokes, tag

def normalize_strokes(strokes, coord_range=1.0):
    if not strokes:
        return strokes
    all_points = [p for s in strokes for p in s]
    if not all_points:
        return strokes
    arr = np.array([np.asarray(p[:2], dtype=np.float32) for p in all_points])
    min_xy, max_xy = arr.min(0), arr.max(0)
    scale = max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1], 1e-6)
    out = []
    for s in strokes:
        new_s = []
        for p in s:
            x = (p[0] - min_xy[0]) / scale * coord_range
            y = (p[1] - min_xy[1]) / scale * coord_range
            new_s.append((x, y) + tuple(p[2:] if len(p) > 2 else []))
        out.append(new_s)
    return out

def apply_rotation_on_strokes(strokes, angle_deg):
    if angle_deg is None or angle_deg == 0:
        return strokes
    theta = np.deg2rad(float(angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    out = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        if len(s_arr) == 0:
            out.append(s_arr)
            continue
        x, y = s_arr[:, 0], s_arr[:, 1]
        xr = c * x - s * y
        yr = s * x + c * y
        new_s = np.stack([xr, yr], axis=1)
        out.append([tuple(row) for row in new_s])
    return out


def bresenham_line(x0: int, y0: int, x1: int, y1: int):
    points = []
    dx, dy = abs(x1 - x0), abs(y1 - y0)
    sx = 1 if x1 > x0 else -1
    sy = 1 if y1 > y0 else -1
    x, y = x0, y0

    if dx > dy:
        err = dx // 2
        while x != x1:
            points.append((x, y))
            err -= dy
            if err < 0:
                y += sy
                err += dx
            x += sx
    else:
        err = dy // 2
        while y != y1:
            points.append((x, y))
            err -= dx
            if err < 0:
                x += sx
                err += dy
            y += sy

    points.append((x1, y1))  # 终点
    return points


def make_disk_struct(radius: int) -> np.ndarray:
    r = radius
    y_g, x_g = np.ogrid[-r:r + 1, -r:r + 1]
    return (x_g ** 2 + y_g ** 2 <= r ** 2)


def strokes_to_image(strokes, img_size: int = 64,
                     thickness: int = 3,
                     coord_range: float = 1.0) -> np.ndarray:

    canvas = np.zeros((img_size, img_size), dtype=bool)

    margin = 2
    draw_size = img_size - 2 * margin
    scale = draw_size / coord_range

    def coord_to_px(v):
        return int(round(v * scale)) + margin

    for stroke in strokes:
        if len(stroke) < 1:
            continue
        pts = np.array(stroke, dtype=np.float32)[:, :2]

        px_list = [
            (
                np.clip(coord_to_px(pt[0]), 0, img_size - 1),
                np.clip(coord_to_px(pt[1]), 0, img_size - 1)
            )
            for pt in pts
        ]

        canvas[px_list[0][1], px_list[0][0]] = True

        for i in range(1, len(px_list)):
            x0, y0 = px_list[i - 1]
            x1, y1 = px_list[i]
            for (lx, ly) in bresenham_line(x0, y0, x1, y1):
                if 0 <= lx < img_size and 0 <= ly < img_size:
                    canvas[ly, lx] = True  # row=y, col=x


    if thickness >= 1:
        disk = make_disk_struct(thickness)
        canvas = binary_dilation(canvas, structure=disk)

    return canvas.astype(np.float32)


def build_one_feature_image(strokes, cfg, angle=None) -> np.ndarray:

    # 1. 旋转增强
    if angle is not None:
        strokes = apply_rotation_on_strokes(strokes, angle)

    # 2. 悬挂归一化
    if cfg.get('use_hanging', False):
        strokes = apply_hanging_on_strokes(strokes, cfg.get('hanging_mode', 'SC'))

    # 3. 坐标归一化到 [0, coord_range]
    strokes = normalize_strokes(strokes, cfg.get('coord_range', 1.0))

    # 4. Bresenham 光栅化 + 加粗
    img = strokes_to_image(
        strokes,
        img_size=cfg.get('img_size', 64),
        thickness=cfg.get('thickness', 3),
        coord_range=cfg.get('coord_range', 1.0)
    )
    return img  # shape: (H, W)


def process_class_image(c_idx, data_dir, is_train, cfg, seed=None):
    raw, _ = read_pot_file(os.path.join(data_dir, f"{c_idx}.pot"))
    if not raw:
        if is_train:
            return [], [], []
        else:
            return ([], [], []), ([], [], [])

    n_tr = int(0.8 * len(raw))
    sel = raw[:n_tr] if is_train else raw[n_tr:]
    rng = np.random.RandomState(int(seed) if seed is not None else 42)

    if is_train:
        imgs, labels, angles = [], [], []
        for s in sel:
            ang = rng.uniform(0, 360)          # 随机旋转增强
            img = build_one_feature_image(s, cfg, angle=ang)
            imgs.append(img)
            labels.append(c_idx - 1)
            angles.append(ang)
        return imgs, labels, angles
    else:
        # te0: 随机角度测试；te30: 30种固定角度测试
        f0, l0, a0 = [], [], []
        f30, l30, a30 = [], [], []
        ang_list = list(range(0, 360, 12))      # 0,12,24,...348 共30个角度
        for s in sel:
            rand_ang = rng.uniform(-180, 180)
            f0.append(build_one_feature_image(s, cfg, angle=rand_ang))
            l0.append(c_idx - 1)
            a0.append(rand_ang)
            for ang in ang_list:
                f30.append(build_one_feature_image(s, cfg, angle=float(ang)))
                l30.append(c_idx - 1)
                a30.append(float(ang))
        return (f0, l0, a0), (f30, l30, a30)



class ImageDataset(Dataset):
    """
    X: list of (H, W) numpy arrays
    """
    def __init__(self, X, y):
        X_arr = np.array(X, dtype=np.float32)        # (N, H, W)
        self.X = torch.tensor(X_arr).unsqueeze(1)    # (N, 1, H, W)
        self.y = torch.tensor(np.array(y), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loaders_image(data_dir, n_cls, cfg, seed=42, mode="train"):
    tr  = ([], [], [])
    te0 = ([], [], [])
    te30= ([], [], [])

    for c in tqdm(range(1, n_cls + 1), desc=f"Build {mode} set"):
        if mode == "train":
            F, L, A = process_class_image(c, data_dir, True, cfg, seed)
            tr[0].extend(F); tr[1].extend(L); tr[2].extend(A)
        (f0, l0, a0), (f30, l30, a30) = process_class_image(c, data_dir, False, cfg)
        te0[0].extend(f0);  te0[1].extend(l0);  te0[2].extend(a0)
        te30[0].extend(f30);te30[1].extend(l30);te30[2].extend(a30)

    bs = cfg.get('batch_size', 64)

    if mode == "train":
        tr_dataset   = ImageDataset(tr[0],   tr[1])
        te30_dataset = ImageDataset(te30[0], te30[1])
        te0_dataset  = ImageDataset(te0[0],  te0[1])

        tr_loader   = DataLoader(tr_dataset,   batch_size=bs,  shuffle=True,  drop_last=False)
        te30_loader = DataLoader(te30_dataset, batch_size=256, shuffle=False)
        te0_loader  = DataLoader(te0_dataset,  batch_size=256, shuffle=False)

        H, W = tr_dataset.X.shape[2], tr_dataset.X.shape[3]
        print(f"Train  samples: {len(tr[0])},  tensor shape: {tuple(tr_dataset.X.shape)}")
        print(f"Test30 samples: {len(te30[0])}, tensor shape: {tuple(te30_dataset.X.shape)}")
        print(f"Image size: {H}×{W}, channels: 1")
        return tr_loader, te30_loader, te0_loader




In [ ]:

class BasicBlock2D(nn.Module):
    def __init__(self, inchannel, outchannel, stride=1):
        super().__init__()
        self.left = nn.Sequential(
            nn.Conv2d(inchannel, outchannel, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(outchannel),
            nn.ReLU(inplace=True),
            nn.Conv2d(outchannel, outchannel, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(outchannel),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or inchannel != outchannel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(inchannel, outchannel, 1, stride=stride, bias=False),
                nn.BatchNorm2d(outchannel),
            )

    def forward(self, x):
        out = self.left(x) + self.shortcut(x)
        return F.relu(out)


class ResNet18_2D(nn.Module):
    def __init__(self, in_channels=1, num_classes=52):
        super().__init__()
        self.inchannel = 64

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2,
                      padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64,  2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(1)         
        self.fc     = nn.Linear(512, num_classes)

        self._init_weights()

    def _make_layer(self, channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers  = []
        for s in strides:
            layers.append(BasicBlock2D(self.inchannel, channels, s))
            self.inchannel = channels
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # x: (B, 1, H, W)
        x = self.stem(x)      # → (B, 64, H,   W)
        x = self.layer1(x)    # → (B, 64, H,   W)
        x = self.layer2(x)    # → (B,128, H/2, W/2)
        x = self.layer3(x)    # → (B,256, H/4, W/4)
        x = self.layer4(x)    # → (B,512, H/8, W/8)
        x = self.gap(x)       # → (B,512, 1,   1)
        x = x.view(x.size(0), -1)   # → (B, 512)
        x = self.fc(x)        # → (B, num_classes)
        return x


In [ ]:
from thop import profile, clever_format

def get_model_metrics(model, img_size: int = 64, in_channels: int = 1):
  
    device = next(model.parameters()).device

    dummy_input = torch.randn(1, in_channels, img_size, img_size).to(device)
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)

    flops = macs * 2.0
    params_str, flops_str = clever_format([params, flops], "%.3f")
    return params_str, flops_str

def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_samples, steps = 0, 0, 0, 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss    += loss.item()
        total_correct += (logits.argmax(-1) == y).sum().item()
        total_samples += y.size(0)
        steps         += 1
    return total_loss / steps, total_correct / total_samples, steps

def evaluate(model, dataloader):
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y in dataloader:
            all_logits.append(model(x.to(device)).cpu().numpy())
            all_targets.append(y.numpy())
    all_logits  = np.concatenate(all_logits,  axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    acc = float(np.mean(np.argmax(all_logits, axis=-1) == all_targets))
    return all_logits, all_targets, acc


def train_loop(model, tr_loader, te30_loader, epochs, save_dir, cfg):
    optimizer = optim.AdamW(model.parameters(), lr=5e-4,
                            weight_decay=cfg['lambd'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    log = {
        "epoch"    : [],
        "loss"     : [],
        "acc"      : [],
        "test_acc" : [],   
        "time"     : [],
        "mem"      : [],
    }
    total_steps, t0 = 0, time.time()

    for ep in range(epochs):
        t_ep = time.time()

        # 训练 
        l, a, steps = train_epoch(model, tr_loader, optimizer, criterion)
        scheduler.step()
        total_steps += steps

        _, _, test_acc = evaluate(model, te30_loader)

        cur_mem    = get_gpu_memory_usage_mb()
        epoch_time = time.time() - t_ep

        if (ep + 1) % cfg['print_every'] == 0 or ep == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"Ep {ep+1:3d}/{epochs} | LR {lr:.2e} | "
                f"Loss {l:.4f} | Train {a:.2%} | "
                f"Test {test_acc:.2%} | "          
                f"Mem {cur_mem:.0f}MB | {epoch_time:.1f}s"
            )

        log["epoch"].append(ep + 1)
        log["loss"].append(l)
        log["acc"].append(a * 100)
        log["test_acc"].append(test_acc * 100)     
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)

        if save_dir and cfg.get('save_ckpt_every') and \
                (ep + 1) % cfg['save_ckpt_every'] == 0:
            os.makedirs(os.path.join(save_dir, "ckpt"), exist_ok=True)
            torch.save(model.state_dict(),
                       os.path.join(save_dir, "ckpt", f"ep{ep+1}.pth"))

    total_time = time.time() - t0
    stats = {
        "total_time"          : total_time,
        "total_steps"         : total_steps,
        "avg_mem"             : float(np.mean(log["mem"])) if log["mem"] else 0.0,
        "time_per_1000_steps" : total_time / total_steps * 1000,
    }
    return model, log, stats


def run_experiment(data_dir, n_cls, save_dir, cfg):
    seeds = cfg.pop('seed_list')
    print(f"Seeds: {seeds}")

    tr_loader, te30_loader, te0_loader = make_loaders_image(
        data_dir, n_cls, cfg, seed=42, mode="train"
    )

    results        = []
    metrics_printed = False
    params_str = flops_str = "N/A"   

    for s in seeds:
        print(f"\n{'='*60}\nSeed {s}\n{'='*60}")
        sd = os.path.join(save_dir, f"seed_{s}")

        if os.path.exists(os.path.join(sd, "_COMPLETED")):
            print("Skip (already done).")
            # 如果已完成，尝试读取summary.json获取结果
            try:
                with open(os.path.join(sd, "summary.json"), "r") as f:
                    summary = json.load(f)
                # 需要重新加载模型来获取logits
                model = ResNet18_2D(in_channels=1, num_classes=n_cls).to(device)
                model.load_state_dict(torch.load(os.path.join(sd, "model.pth")))
                l30, t30, a30 = evaluate(model, te30_loader)
                results.append({
                    "acc": a30, 
                    "logits": l30, 
                    "targets": t30,
                    "time_per_1000_steps": summary.get("time_per_1000_steps", 0),
                    "avg_mem": summary.get("avg_mem", 0),
                    "total_time": summary.get("total_time", 0)
                })
            except:
                pass
            continue

        os.makedirs(sd, exist_ok=True)
        torch.manual_seed(s)
        torch.cuda.manual_seed_all(s)

        model = ResNet18_2D(in_channels=1, num_classes=n_cls).to(device)

        if not metrics_printed:
            params_str, flops_str = get_model_metrics(
                model, img_size=cfg['img_size'], in_channels=1
            )
            print(f"{'─'*40}")
            print(f"  Model  : ResNet18_2D")
            print(f"  Params : {params_str}")
            print(f"  FLOPs  : {flops_str}  "
                  f"(input {cfg['img_size']}×{cfg['img_size']}×1)")
            print(f"{'─'*40}")
            metrics_printed = True

        model, log, stats = train_loop(
            model, tr_loader, te30_loader,
            cfg['total_epochs'], sd, cfg
        )

        # 保存前100个epoch的精度
        epochs_to_save = min(100, len(log['test_acc']))
        if epochs_to_save > 0:
            df_acc = pd.DataFrame({
                "Epoch"         : log['epoch'][:epochs_to_save],
                "Train_Accuracy": log['acc'][:epochs_to_save],
                "Test_Accuracy" : log['test_acc'][:epochs_to_save],
            })
            csv_path = os.path.join(sd, f"first100_acc_seed{s}.csv")
            df_acc.to_csv(csv_path, index=False)
            print(f"前 {epochs_to_save} 个 Epoch 精度已保存: {csv_path}")

        # 评估
        l30, t30, a30 = evaluate(model, te30_loader)
        print(f"\n[Seed {s}] 30x Test Acc: {a30:.4f}")
        print(f"  Time per 1000 steps: {stats['time_per_1000_steps']:.2f}s")
        print(f"  Avg GPU Memory: {stats['avg_mem']:.2f}MB")

        torch.save(model.state_dict(), os.path.join(sd, "model.pth"))
        with open(os.path.join(sd, "summary.json"), "w") as f:
            json.dump(
                {
                    "acc"                 : a30,
                    "seed"                : s,
                    "params"              : params_str,
                    "flops"               : flops_str,
                    **stats
                },
                f, indent=2
            )
        open(os.path.join(sd, "_COMPLETED"), "w").close()

        results.append({
            "acc": a30, 
            "logits": l30, 
            "targets": t30,
            "time_per_1000_steps": stats['time_per_1000_steps'],
            "avg_mem": stats['avg_mem'],
            "total_time": stats['total_time']
        })
        print(f"Seed {s} done. Acc={a30:.4f}")

    if results:
        accs = [r["acc"] for r in results]
        times_per_1k = [r["time_per_1000_steps"] for r in results]
        avg_mems = [r["avg_mem"] for r in results]
        total_times = [r["total_time"] for r in results]
        
        print(f"\n{'='*60}")
        print(f"FINAL RESULTS ({len(results)} seeds)")
        print(f"{'='*60}")
        print(f"Accuracy:")
        print(f"  Mean ± Std : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        print(f"\nTraining Efficiency:")
        print(f"  Time per 1000 steps : {np.mean(times_per_1k):.2f} ± {np.std(times_per_1k):.2f} s")
        print(f"  Avg GPU Memory      : {np.mean(avg_mems):.2f} ± {np.std(avg_mems):.2f} MB")
        print(f"  Total Training Time : {np.mean(total_times)/60:.1f} ± {np.std(total_times)/60:.1f} min")
    
        all_logits = np.stack([r["logits"] for r in results])  
        targets    = results[0]["targets"]
    
        # Soft Voting
        probs    = F.softmax(torch.tensor(all_logits), dim=-1).numpy()  
        avg_prob = probs.mean(axis=0)                              
        soft_acc = float(np.mean(np.argmax(avg_prob, axis=-1) == targets))
    
        # Hard Voting
        all_preds  = np.argmax(all_logits, axis=-1)         
        n_samples  = all_preds.shape[1]
        hard_preds = np.zeros(n_samples, dtype=np.int64)
        for i in range(n_samples):
            votes          = np.bincount(all_preds[:, i])
            hard_preds[i]  = np.argmax(votes)
        hard_acc = float(np.mean(hard_preds == targets))
    
        print(f"\nEnsemble Results:")
        print(f"  Soft Voting Acc : {soft_acc:.4f}")
        print(f"  Hard Voting Acc : {hard_acc:.4f}")
        print("="*60)



In [20]:

if __name__ == "__main__":
    DATA_DIR   = "RotFreeData/Radicals52" # EngUpper26 / Digits10 / Radicals52
    SAVE_DIR   = "Model/Resnet18/Resnet18_Radicals52/"
    NUM_CLASSES = 52

    CFG = {
        "img_size"   : 64,    # 图像分辨率 64×64
        "thickness"  : 3,     # 笔画加粗半径，直径≈7px
        "coord_range": 1.0,

        "use_hanging"  : True,
        "hanging_mode" : "SC",

        "total_epochs"  : 200,
        "batch_size"    : 64,
        "lambd"         : 1e-4,
        "print_every"   : 1,
        "save_ckpt_every": 0,
        "seed_list"     : [1011, 1012, 1013, 1014, 1015],
    }

    run_experiment(DATA_DIR, NUM_CLASSES, SAVE_DIR, CFG)

Seeds: [1011, 1012, 1013, 1014, 1015]


Build train set: 100%|██████████████████████████████████████████████████████████████████| 52/52 [01:25<00:00,  1.64s/it]


Train  samples: 12401,  tensor shape: (12401, 1, 64, 64)
Test30 samples: 93510, tensor shape: (93510, 1, 64, 64)
Image size: 64×64, channels: 1

Seed 1011
────────────────────────────────────────
  Model  : ResNet18_2D
  Params : 11.194M
  FLOPs  : 1.113G  (input 64×64×1)
────────────────────────────────────────
Ep   1/200 | LR 5.00e-04 | Loss 1.3954 | Train 62.13% | Test 76.68% | Mem 191MB | 11.5s
Ep   2/200 | LR 5.00e-04 | Loss 0.5177 | Train 84.64% | Test 81.46% | Mem 191MB | 11.5s
Ep   3/200 | LR 5.00e-04 | Loss 0.3690 | Train 88.92% | Test 83.83% | Mem 191MB | 11.0s
Ep   4/200 | LR 5.00e-04 | Loss 0.2826 | Train 91.35% | Test 86.53% | Mem 191MB | 11.4s
Ep   5/200 | LR 4.99e-04 | Loss 0.2149 | Train 93.12% | Test 87.68% | Mem 191MB | 11.6s
Ep   6/200 | LR 4.99e-04 | Loss 0.1743 | Train 94.12% | Test 84.76% | Mem 191MB | 11.6s
Ep   7/200 | LR 4.98e-04 | Loss 0.1350 | Train 95.56% | Test 87.49% | Mem 191MB | 11.4s
Ep   8/200 | LR 4.98e-04 | Loss 0.1035 | Train 96.57% | Test 86.72% | 